In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import TrainableFidelityQuantumKernel
from qiskit_machine_learning.kernels.algorithms import QuantumKernelTrainer
from qiskit_machine_learning.optimizers import SPSA

# ✅ CORREÇÃO 1: usar QSVR nativo em vez de SVR(kernel=evaluate)
from qiskit_machine_learning.algorithms import QSVR

# ==========================
# LOAD DATA
# ==========================
dataset = pd.read_csv("../dataset/riemann_features.csv")
features = [
    "z_co_gram_lag_2", "z_gram", "z_gram_lag_1", "d_lag_13",
    "z_co_gram_lag_3", "z_co_gram_lag_1", "d_lag_14",
    "d_lag_1", "z_gram_lag_2", "d_lag_17"
]
X = dataset[features].values[:2000]
y = dataset["distance"].values[:2000]

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# ==========================
# SCALE  (range π maximiza separação angular no espaço de Hilbert)
# ==========================
scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

n_qubits = X_train.shape[1]  # 10 — sem PCA, respeitando Rank Aggregation

# ==========================
# FEATURE MAP
# ==========================
x     = ParameterVector("x", n_qubits)
theta = ParameterVector("θ", n_qubits * 3)

qc = QuantumCircuit(n_qubits)

# Camada treinável (θ) antes do encoding
k = 0
for layer in range(3):
    for i in range(n_qubits):
        qc.rz(theta[k], i)
        k += 1
    for i in range(n_qubits - 1):
        qc.cx(i, i + 1)
    qc.cx(n_qubits - 1, 0)  # entanglement circular

# Camada de encoding (x)
for i in range(n_qubits):
    qc.ry(x[i], i)

# ==========================
# KERNEL
# ==========================
sampler  = StatevectorSampler()
fidelity = ComputeUncompute(sampler=sampler)

quantum_kernel = TrainableFidelityQuantumKernel(
    feature_map=qc,
    fidelity=fidelity,
    training_parameters=theta
)

# ==========================
# TRAIN KERNEL
# ✅ CORREÇÃO 2: o QuantumKernelTrainer com svc_loss precisa de labels
#    discretos. Binarizamos y apenas para o trainer achar os θ ótimos.
#    O kernel treinado é agnóstico ao tipo de tarefa — os parâmetros θ
#    aprendidos valem tanto para classificação quanto para regressão.
# ==========================
y_binary = (y_train > np.median(y_train)).astype(int)  # 0 / 1 para o trainer

optimizer = SPSA(maxiter=50, learning_rate=0.05, perturbation=0.05)

kernel_trainer = QuantumKernelTrainer(
    quantum_kernel=quantum_kernel,
    loss="svc_loss",
    optimizer=optimizer,
    initial_point=[np.pi / 2] * len(theta)
)

print("Training Quantum Kernel...")
kernel_result    = kernel_trainer.fit(X_train, y_binary)   # ← labels discretos
optimized_kernel = kernel_result.quantum_kernel

# ==========================
# QSVR — regressão com y contínuo
# ✅ CORREÇÃO 3: QSVR aceita quantum_kernel= diretamente, sem gambiarra
#    de kernel=evaluate, e não tem o bug do label type
# ==========================
qsvr = QSVR(
    quantum_kernel=optimized_kernel,
    C=10,
    epsilon=0.001
)

print("Training QSVR...")
qsvr.fit(X_train, y_train)    # ← y contínuo aqui, sem problema
pred = qsvr.predict(X_test)

# ==========================
# METRICS
# ==========================
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2   = r2_score(y_test, pred)

print("\n===== Quantum Neural Kernel (QSVR) =====")
print(f"RMSE: {rmse:.6f}")
print(f"R2:   {r2:.6f}")

Training Quantum Kernel...


In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Kernel

from qiskit.circuit.library import pauli_feature_map
from qiskit.primitives import StatevectorSampler

from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

# ==========================
# LOAD DATA
# ==========================

dataset = pd.read_csv("../dataset/riemann_features.csv")

features = [
    "z_co_gram_lag_2",
    "z_gram",
    "z_gram_lag_1",
    "d_lag_13",
    "z_co_gram_lag_3",
    "z_co_gram_lag_1",
    "d_lag_14",
    "d_lag_1",
    "z_gram_lag_2",
    "d_lag_17"
]

X = dataset[features].values
y = dataset["distance"].values

X = X[:2000]
y = y[:2000]

split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

# ==========================
# SCALE
# ==========================

scaler = MinMaxScaler(feature_range=(-1,1))

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================
# PCA (reduzir qubits)
# ==========================


n_qubits = X_train.shape[1]

# ==========================
# QUANTUM KERNEL
# ==========================

feature_map = pauli_feature_map(
    feature_dimension=n_qubits,
    reps=4,
    entanglement="circular",
    paulis=["Z","ZZ","ZX"]
)

sampler = StatevectorSampler()

fidelity = ComputeUncompute(sampler=sampler)

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map,
    fidelity=fidelity
)

# ==========================
# WRAPPER PARA SKLEARN
# ==========================

class QuantumKernelWrapper(Kernel):

    def __init__(self, quantum_kernel):
        self.quantum_kernel = quantum_kernel

    def __call__(self, X, Y=None, eval_gradient=False):

        if Y is None:
            K = self.quantum_kernel.evaluate(X)
        else:
            K = self.quantum_kernel.evaluate(X, Y)

        if eval_gradient:
            return K, np.zeros((K.shape[0], K.shape[1], 1))

        return K

    def diag(self, X):
        return np.ones(X.shape[0])

    def is_stationary(self):
        return False

kernel = QuantumKernelWrapper(quantum_kernel)

# ==========================
# GAUSSIAN PROCESS
# ==========================

gpr = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    normalize_y=True
)

print("Treinando Quantum Gaussian Process...")

gpr.fit(X_train, y_train)

pred = gpr.predict(X_test)

# ==========================
# MÉTRICAS
# ==========================

rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("\n===== Quantum Gaussian Process =====")
print("RMSE:", rmse)
print("R2:", r2)